In [2]:
from pathlib import Path
import numpy as np
import rasterio
import xarray as xr
from rasterio.warp import transform_bounds

LANDSAT_TIF = Path(r"../Timor_part1/landsat_sst_outputs/LC08_L2SP_110066_20200113_20200824_02_T1_SST_C.tif")
VIIRS_NC = Path(r"../Timor_part1/L3S_STAR/raw/2020-01/PM/20200113120000-STAR-L3S_GHRSST-SSTsubskin-LEO_PM_D-ACSPO_V2.81-v02.0-fv01.0.nc")

BUFFER_LIST = [0, 10, 20]

def get_first_existing_var(ds, candidates):
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    return None

with rasterio.open(LANDSAT_TIF) as ds:
    west, south, east, north = transform_bounds(
        ds.crs, "EPSG:4326", *ds.bounds, densify_pts=21
    )
    landsat_width = ds.width
    landsat_height = ds.height

lon_per_px = (east - west) / landsat_width
lat_per_px = (north - south) / landsat_height

print("LANDSAT bounds:")
print("west :", west)
print("south:", south)
print("east :", east)
print("north:", north)

with xr.open_dataset(VIIRS_NC, decode_timedelta=False) as ds:
    lon_var = get_first_existing_var(ds, ["lon", "longitude"])
    lat_var = get_first_existing_var(ds, ["lat", "latitude"])

    if lon_var is None or lat_var is None:
        raise RuntimeError("Cannot find lon/lat variables in VIIRS nc")

    lon = np.asarray(ds[lon_var].values)
    lat = np.asarray(ds[lat_var].values)

lon = np.where(lon > 180, lon - 360, lon)

vwest = float(np.nanmin(lon))
veast = float(np.nanmax(lon))
vsouth = float(np.nanmin(lat))
vnorth = float(np.nanmax(lat))

print("\nVIIRS NC bounds:")
print("west :", vwest)
print("south:", vsouth)
print("east :", veast)
print("north:", vnorth)

def check_coverage(buffer_px):
    buf_lon = buffer_px * abs(lon_per_px)
    buf_lat = buffer_px * abs(lat_per_px)

    req_west = west - buf_lon
    req_east = east + buf_lon
    req_south = south - buf_lat
    req_north = north + buf_lat

    covered = (
        vwest <= req_west and
        veast >= req_east and
        vsouth <= req_south and
        vnorth >= req_north
    )

    print("\n" + "-" * 60)
    print(f"BUFFER = {buffer_px}px")
    print("Required:")
    print(req_west, req_south, req_east, req_north)
    print("Covered by VIIRS NC?", covered)

    if not covered:
        if vwest > req_west:
            print("Missing WEST")
        if veast < req_east:
            print("Missing EAST")
        if vsouth > req_south:
            print("Missing SOUTH")
        if vnorth < req_north:
            print("Missing NORTH")

for b in BUFFER_LIST:
    check_coverage(b)

LANDSAT bounds:
west : 124.13763656435658
south: -9.738524571993645
east : 126.22954328226425
north: -7.619802939799301


/home/mingyue/apps/miniforge3/envs/sst/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)



VIIRS NC bounds:
west : -179.99000549316406
south: -89.98999786376953
east : 179.99000549316406
north: 89.98999786376953

------------------------------------------------------------
BUFFER = 0px
Required:
124.13763656435658 -9.738524571993645 126.22954328226425 -7.619802939799301
Covered by VIIRS NC? True

------------------------------------------------------------
BUFFER = 10px
Required:
124.13489523724624 -9.741251018567052 126.23228460937459 -7.617076493225894
Covered by VIIRS NC? True

------------------------------------------------------------
BUFFER = 20px
Required:
124.13215391013588 -9.743977465140459 126.23502593648494 -7.614350046652487
Covered by VIIRS NC? True
